# Zeeman Analysis — Quick Workflow

**Two-stage workflow:** preprocessing (run once) → analysis (fast, repeatable).

1. **Preprocess** (expensive): Run only when you have new images or need to recompute radial profiles.
2. **Analyze** (fast): Loads cached `r_arr.npy`, `I_r_arr.npy` and computes fractional frequency shifts + Bohr magneton.

## 1. Configuration

In [ ]:
from pathlib import Path
import numpy as np

data_dir = Path('.')
cache_dir = data_dir

# Image paths (in order of increasing B field) — only needed for preprocessing
image_paths = [
    data_dir / 'vdc_20_splitting_measurements.png',
    data_dir / 'vdc_22.5_splitting_measurements.png',
    data_dir / 'vdc_25_splitting_measurements.png',
    data_dir / 'vdc_27.5_splitting_measurements.png',
    data_dir / 'vdc_29_splitting_measurements.png',
]

# Magnetic fields in kG (kilogauss) — must match image order
B_fields_kG = np.array([6, 6.7, 7.3, 8.05, 9.3], dtype=float)

# Etalon thickness: 8.11 mm
etalon_thickness_cm = 0.811

## 2. Preprocessing (run once)

Skip this cell if `r_arr.npy` and `I_r_arr.npy` already exist in `cache_dir`.

In [ ]:
from zeeman_analysis_pipeline import load_or_compute_profiles

# Loads cache if present; otherwise computes and saves
I_r_arr, r_arr, centers = load_or_compute_profiles(
    image_paths=[str(p) for p in image_paths],
    cache_dir=cache_dir,
    force_recompute=False,  # Set True to recompute
    nbins=2000,
)
print(f'Profiles loaded: {len(r_arr)}')

## 3. Analysis (fast)

Loads cached profiles and computes fractional frequency shifts + Bohr magneton. No image loading or center-finding.

In [ ]:
from zeeman_analyze import AnalyzeConfig, run_analysis, plot_results

config = AnalyzeConfig(
    cache_dir=cache_dir,
    etalon_thickness_cm=etalon_thickness_cm,
    B_fields=B_fields_kG,
    B_scale=1000.0,  # kG → G
    fsr_mode='r2',  # r² scaling for Fabry-Perot accuracy
    use_triplet=True,
    out_ring_csv=cache_dir / 'zeeman_ringwise_results.csv',
    out_summary_csv=cache_dir / 'zeeman_summary_results.csv',
    out_plot=cache_dir / 'zeeman_vs_B.png',
)

df_ring, df_summary, magneton = run_analysis(config)

In [ ]:
# Save results
df_ring.to_csv(config.out_ring_csv, index=False)
df_summary.to_csv(config.out_summary_csv, index=False)
if config.out_plot:
    plot_results(df_summary, magneton, config.out_plot)
print('Saved:', config.out_ring_csv, config.out_summary_csv, config.out_plot)

In [ ]:
# Bohr magneton result
print('Slope (free fit)     =', magneton['slope_cm_inv_per_G'], 'cm⁻¹/G')
print('Slope (through orig) =', magneton['slope_through_origin'], 'cm⁻¹/G')
print('Estimated μ_B        =', f"{magneton['mu_B_J_per_T']:.4e} J/T")
print('Accepted μ_B         =', f"{magneton['accepted_mu_B']:.4e} J/T")

In [ ]:
# Fractional shifts by ring and B field
import pandas as pd
df_ring[df_ring['success']][['B_g', 'ring_idx', 'frac_shift', 'delta_wn_cm_inv']].head(15)